In [1]:
# Cell 1: Install Colab dependencies
!pip -q install fastapi uvicorn python-multipart faster-whisper openai pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 17.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 17.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 55.4 MB/s eta 0:00:0000:0100:01m


## Cell 2: Configure ngrok token and open tunnel
Run this cell after replacing YOUR_NGROK_AUTHTOKEN if needed.

In [3]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = "2g6j9ouU0iNsHZei0uvceev6Fdx_3s3NeRbboEK3RfCGHmYqq"
public_url = ngrok.connect(7860, "http").public_url

print("COLAB_AI_BASE_URL=" + public_url)
print("COLAB_VOICE_PATH=/voice")
print("COLAB_AI_API_KEY=")
print("COLAB_TIMEOUT_SECONDS=20")

COLAB_AI_BASE_URL=https://ff78-35-231-160-252.ngrok-free.app                                        
COLAB_VOICE_PATH=/voice
COLAB_AI_API_KEY=
COLAB_TIMEOUT_SECONDS=20


In [6]:
# Cell 4: Write the Colab voice server file (CPU-safe, auto device)
server_code = '''
import json
import os
import tempfile
from typing import Dict, Any

from fastapi import FastAPI, UploadFile, File, Form
from faster_whisper import WhisperModel
from openai import OpenAI

app = FastAPI(title='Suraksha Colab Voice Server')

WHISPER_MODEL = os.getenv('WHISPER_MODEL', 'small')
# Use cuda if available, else cpu
try:
    import torch
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    _compute = 'float16' if _device == 'cuda' else 'int8'
except ImportError:
    _device, _compute = 'cpu', 'int8'

print(f'Loading Whisper model={WHISPER_MODEL} device={_device} compute={_compute}')
whisper = WhisperModel(WHISPER_MODEL, device=_device, compute_type=_compute)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
llm_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

@app.get('/health')
def health() -> Dict[str, Any]:
    return {'ok': True, 'whisper_model': WHISPER_MODEL, 'device': _device, 'llm': bool(llm_client)}

@app.post('/voice')
async def voice(file: UploadFile = File(...), role: str = Form('citizen'), language: str = Form(''), context: str = Form('{}')):
    raw = await file.read()
    if not raw:
        return {'error': 'empty_audio', 'response': None}
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as temp:
        temp.write(raw)
        audio_path = temp.name
    segments, info = whisper.transcribe(audio_path, language=None if not language else language[:2])
    transcript = ' '.join([seg.text.strip() for seg in segments]).strip()
    if not transcript:
        return {'error': 'stt_failed', 'response': None, 'transcript': ''}
    answer = _generate_answer(role, transcript, context)
    return {
        'transcript': transcript,
        'response': answer,
        'detected_language': (info.language if info else None) or language or 'unknown',
        'usage': {'provider': 'colab', 'device': _device, 'role': role},
        'error': None,
    }

def _generate_answer(role: str, transcript: str, context_json: str) -> str:
    if not llm_client:
        return f'I heard: {transcript}. Configure OPENAI_API_KEY for richer responses.'
    try:
        parsed_context = json.loads(context_json) if context_json else {}
    except Exception:
        parsed_context = {}
    system_prompt = (
        'You are a concise disaster safety assistant for India. '
        f'User role: {role}. Give practical, actionable safety guidance.'
    )
    response = llm_client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Transcript: {transcript}\\nContext: {json.dumps(parsed_context, ensure_ascii=True)}'},
        ],
        temperature=0.4,
        max_tokens=300,
    )
    return response.choices[0].message.content or 'I am here to help.'
'''

with open('colab_voice_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)
print('Created colab_voice_server.py')

Created colab_voice_server.py


In [7]:
# Cell 5: Start the voice server in a background thread so this cell completes
import threading, uvicorn, importlib, sys

# Reload server module if already imported
if 'colab_voice_server' in sys.modules:
    del sys.modules['colab_voice_server']

import colab_voice_server as _srv

config = uvicorn.Config(_srv.app, host='0.0.0.0', port=7860, log_level='info')
server = uvicorn.Server(config)

t = threading.Thread(target=server.run, daemon=True)
t.start()

import time; time.sleep(3)  # wait for startup
print('Voice server started on port 7860')

Loading Whisper model=small device=cpu compute=int8


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
INFO:     Started server process [968]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:7860 (Press CTRL+C to quit)


Voice server started on port 7860


In [8]:
# Cell 6: Verify server is healthy
import urllib.request, json
try:
    req = urllib.request.Request('http://localhost:7860/health')
    with urllib.request.urlopen(req, timeout=5) as r:
        data = json.loads(r.read())
    print('Server health:', data)
    print()
    print('COLAB_AI_BASE_URL=' + public_url)
    print('COLAB_VOICE_PATH=/voice')
    print('COLAB_AI_API_KEY=')
    print('COLAB_TIMEOUT_SECONDS=20')
except Exception as e:
    print('Server not ready yet:', e)

INFO:     127.0.0.1:45052 - "GET /health HTTP/1.1" 200 OK
Server health: {'ok': True, 'whisper_model': 'small', 'device': 'cpu', 'llm': False}

COLAB_AI_BASE_URL=https://ff78-35-231-160-252.ngrok-free.app
COLAB_VOICE_PATH=/voice
COLAB_AI_API_KEY=
COLAB_TIMEOUT_SECONDS=20
